# Progetto: Timed GNN per la risoluzione di puzzle scacchistici (Lichess)
### Fase 3 — Modello GNN e training loop

Questo notebook implementa il **Timed Graph Neural Network** e il ciclo di addestramento
(metodologia, sezione 2 del PDF), in assenza della libreria custom di Wong/Damiani, usando
**PyTorch Geometric** come da requisito "Resources Required" del progetto.

Il notebook è **autosufficiente**: ricostruisce in forma compatta le funzioni di encoding già
sviluppate nella Fase 2, così puoi eseguirlo anche in una sessione Colab separata (basta avere i
file Parquet della Fase 1 in `/content/data/`).

**Cosa produce:**
1. Definizione del modello `TimedGNN` (message passing con `GINEConv`, policy head sugli archi "mossa legale")
2. Training loop con métriche (loss, accuracy@1, accuracy per valore di `n`)
3. Un modello **con timing** e uno **senza timing** (per l'ablation study, Fase 5 del progetto)
4. Checkpoint dei modelli migliori, pronti per il confronto con l'LLM nella Fase 4


## 0. Setup

In [ ]:
!pip install -q python-chess torch_geometric

import os
import time
import numpy as np
import pandas as pd
import chess
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv
from torch_geometric.utils import softmax as pyg_softmax

DATA_DIR = "/content/data"
CKPT_DIR = "/content/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


## 1. Recap Fase 1+2 — caricamento dati e funzioni di encoding

(Versione condensata di quanto sviluppato nei notebook precedenti, per rendere questo notebook
eseguibile in autonomia.)


In [ ]:
required_files = ["puzzles_train.parquet", "puzzles_val.parquet", "puzzles_test.parquet"]
missing = [f for f in required_files if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    raise FileNotFoundError(
        f"File mancanti: {missing}. Esegui prima 01_data_pipeline.ipynb, "
        f"oppure carica i Parquet in {DATA_DIR}."
    )

df_train = pd.read_parquet(os.path.join(DATA_DIR, "puzzles_train.parquet"))
df_val = pd.read_parquet(os.path.join(DATA_DIR, "puzzles_val.parquet"))
df_test = pd.read_parquet(os.path.join(DATA_DIR, "puzzles_test.parquet"))
print(f"Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}")


In [ ]:
PIECE_TYPES = [chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING]
NUM_PIECE_FEATS = len(PIECE_TYPES) * 2 + 1
NODE_FEAT_DIM = NUM_PIECE_FEATS + 2 + 1  # + (file,rank) + timing

def square_node_features(board, square):
    feat = [0.0] * NUM_PIECE_FEATS
    piece = board.piece_at(square)
    if piece is None:
        feat[-1] = 1.0
    else:
        idx = PIECE_TYPES.index(piece.piece_type)
        color_offset = 0 if piece.color == chess.WHITE else len(PIECE_TYPES)
        feat[idx + color_offset] = 1.0
    return feat + [chess.square_file(square) / 7.0, chess.square_rank(square) / 7.0]

def build_edges(board):
    edge_src, edge_dst, edge_type = [], [], []
    legal_moves = list(board.legal_moves)
    for move in legal_moves:
        edge_src.append(move.from_square); edge_dst.append(move.to_square); edge_type.append(0)
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece is None:
            continue
        for target_sq in board.attacks(square):
            edge_src.append(square); edge_dst.append(target_sq); edge_type.append(1)

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    edge_type_t = torch.tensor(edge_type, dtype=torch.long)
    edge_attr = F.one_hot(edge_type_t, num_classes=2).float()
    return edge_index, edge_attr, legal_moves

def get_position_and_target(fen, moves_str):
    board = chess.Board(fen)
    moves = moves_str.split()
    if len(moves) >= 2:
        board.push(chess.Move.from_uci(moves[0]))
        target_uci = moves[1]
    else:
        target_uci = moves[0]
    return board, chess.Move.from_uci(target_uci)

class ChessPuzzleGraphDataset(Dataset):
    def __init__(self, df, include_timing=True):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.include_timing = include_timing

    def len(self):
        return len(self.df)

    def get(self, idx):
        row = self.df.iloc[idx]
        board, target_move = get_position_and_target(row["FEN"], row["Moves"])
        node_feats = torch.tensor([square_node_features(board, sq) for sq in chess.SQUARES], dtype=torch.float)

        if self.include_timing:
            t = min(float(row["simulated_think_time_sec"]), 15.0) / 15.0
        else:
            t = 0.0
        timing_col = torch.full((node_feats.size(0), 1), t, dtype=torch.float)
        node_feats = torch.cat([node_feats, timing_col], dim=1)

        edge_index, edge_attr, legal_moves = build_edges(board)
        try:
            target_local_idx = legal_moves.index(target_move)
        except ValueError:
            target_local_idx = -1

        data = Data(x=node_feats, edge_index=edge_index, edge_attr=edge_attr)
        data.y = torch.tensor([target_local_idx], dtype=torch.long)
        data.num_candidate_moves = torch.tensor([len(legal_moves)], dtype=torch.long)
        data.mate_n = torch.tensor([int(row["mate_n"])], dtype=torch.long)
        return data

print("Funzioni di encoding pronte. NODE_FEAT_DIM =", NODE_FEAT_DIM)


## 2. Il modello `TimedGNN`

- **Encoder**: alcuni layer di message passing `GINEConv` (supportano nativamente `edge_attr`,
  quindi il modello distingue tra archi "mossa legale" e archi "attacco")
- **Policy head**: per ogni arco "mossa legale", concatena gli embedding di nodo sorgente/destinazione
  e produce uno score; la mossa scelta è quella con score più alto **tra le mosse legali della stessa
  posizione** (softmax raggruppata per grafo, non globale sul batch)

`include_timing` a livello di **dataset** decide se la feature di timing è presente o azzerata; il
modello stesso non ha bisogno di saperlo esplicitamente, il che rende pulito il confronto per
l'ablation study (stessa architettura, dati diversi).


In [ ]:
class TimedGNN(nn.Module):
    def __init__(self, node_feat_dim, edge_feat_dim=2, hidden_dim=128, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(node_feat_dim, hidden_dim)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)
            )
            self.convs.append(GINEConv(mlp, edge_dim=edge_feat_dim))
            self.norms.append(nn.LayerNorm(hidden_dim))
        self.dropout = nn.Dropout(dropout)
        self.edge_scorer = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1)
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch_vec = data.x, data.edge_index, data.edge_attr, data.batch
        h = self.input_proj(x)
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index, edge_attr)
            h_new = norm(h_new)
            h = F.relu(h_new) + h  # residual
            h = self.dropout(h)

        candidate_mask = edge_attr[:, 0] == 1
        cand_src = edge_index[0][candidate_mask]
        cand_dst = edge_index[1][candidate_mask]

        edge_repr = torch.cat([h[cand_src], h[cand_dst]], dim=1)
        scores = self.edge_scorer(edge_repr).squeeze(-1)
        edge_batch = batch_vec[cand_src]

        return scores, edge_batch


## 3. Funzioni di loss, accuracy e training/valutazione

Loss: negative log-likelihood sulla probabilità (softmax raggruppata per grafo) assegnata alla mossa
corretta. Accuracy: percentuale di posizioni in cui la mossa con score più alto è quella corretta
(accuracy@1), calcolata anche **per valore di `n`** per rispondere alla research question 1.


In [ ]:
def compute_targets_and_offsets(batch):
    num_cand = batch.num_candidate_moves
    offsets = torch.cat([torch.zeros(1, dtype=torch.long, device=num_cand.device),
                          torch.cumsum(num_cand, dim=0)[:-1]])
    correct_global_idx = offsets + batch.y.squeeze(-1)
    return correct_global_idx

def loss_and_metrics(scores, edge_batch, correct_global_idx, mate_n_per_graph, num_graphs, eps=1e-12):
    probs = pyg_softmax(scores, edge_batch)
    loss = -torch.log(probs[correct_global_idx] + eps).mean()

    correct_flags = torch.zeros(num_graphs, dtype=torch.bool, device=scores.device)
    for g in range(num_graphs):
        mask = edge_batch == g
        idx_in_group = scores[mask].argmax().item()
        global_idx = mask.nonzero(as_tuple=True)[0][idx_in_group].item()
        correct_flags[g] = (global_idx == correct_global_idx[g].item())

    acc = correct_flags.float().mean().item()
    return loss, acc, correct_flags

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_correct, total_graphs = 0.0, 0, 0
    per_n_correct, per_n_total = {}, {}

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for batch in loader:
            batch = batch.to(device)
            scores, edge_batch = model(batch)
            correct_global_idx = compute_targets_and_offsets(batch)
            loss, acc, correct_flags = loss_and_metrics(
                scores, edge_batch, correct_global_idx, batch.mate_n, batch.num_graphs
            )

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            bs = batch.num_graphs
            total_loss += loss.item() * bs
            total_correct += acc * bs
            total_graphs += bs

            mate_n_vals = batch.mate_n.squeeze(-1).cpu().tolist()
            for n_val, is_correct in zip(mate_n_vals, correct_flags.cpu().tolist()):
                per_n_total[n_val] = per_n_total.get(n_val, 0) + 1
                per_n_correct[n_val] = per_n_correct.get(n_val, 0) + int(is_correct)

    avg_loss = total_loss / total_graphs
    avg_acc = total_correct / total_graphs
    per_n_acc = {n: per_n_correct[n] / per_n_total[n] for n in sorted(per_n_total)}
    return avg_loss, avg_acc, per_n_acc


## 4. Iperparametri

Valori di partenza in linea con la metodologia del progetto (learning rate 1e-3, 3-5 layer, hidden
dim 128-256, batch size 32-128, 50-100 epoche). Su Colab con GPU gratuita, per un primo esperimento
consigliamo di partire da un **sottoinsieme** dei dati (`SUBSET_SIZE`) e poche epoche, per validare
che tutto funzioni prima di lanciare il training completo (che con centinaia di migliaia di puzzle e
50-100 epoche può richiedere diverse ore).


In [ ]:
HIDDEN_DIM = 128
NUM_LAYERS = 4
DROPOUT = 0.1
LEARNING_RATE = 1e-3
BATCH_SIZE = 64
NUM_EPOCHS = 10          # aumentare a 50-100 per il training completo (metodologia)
SUBSET_SIZE = 20000      # None per usare tutto il dataset; utile ridurlo per i primi test end-to-end

def maybe_subset(df, n):
    if n is None or n >= len(df):
        return df
    return df.sample(n, random_state=SEED).reset_index(drop=True)

df_train_run = maybe_subset(df_train, SUBSET_SIZE)
df_val_run = maybe_subset(df_val, SUBSET_SIZE // 5 if SUBSET_SIZE else None)

print(f"Training su {len(df_train_run):,} puzzle, validazione su {len(df_val_run):,}")


## 5. Training — modello CON timing

In [ ]:
train_ds = ChessPuzzleGraphDataset(df_train_run, include_timing=True)
val_ds = ChessPuzzleGraphDataset(df_val_run, include_timing=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

model_timed = TimedGNN(NODE_FEAT_DIM, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT).to(device)
optimizer = torch.optim.Adam(model_timed.parameters(), lr=LEARNING_RATE)

history_timed = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = -1.0

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc, _ = run_epoch(model_timed, train_loader, optimizer)
    val_loss, val_acc, val_per_n = run_epoch(model_timed, val_loader, optimizer=None)
    dt = time.time() - t0

    history_timed["train_loss"].append(train_loss)
    history_timed["val_loss"].append(val_loss)
    history_timed["train_acc"].append(train_acc)
    history_timed["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model_timed.state_dict(), os.path.join(CKPT_DIR, "timed_gnn_best.pt"))

    print(f"[timed] epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} | {dt:.1f}s")

print(f"\nMiglior val accuracy (con timing): {best_val_acc:.3f} -> salvato in "
      f"{os.path.join(CKPT_DIR, 'timed_gnn_best.pt')}")


## 6. Training — modello SENZA timing (ablation)

Stessa architettura, stessi iperparametri, unica differenza: `include_timing=False` nel dataset
(la feature di timing viene azzerata). Questo isola l'effetto della feature temporale, come richiesto
dall'obiettivo 5 del progetto (Ablation Study).


In [ ]:
train_ds_nt = ChessPuzzleGraphDataset(df_train_run, include_timing=False)
val_ds_nt = ChessPuzzleGraphDataset(df_val_run, include_timing=False)

train_loader_nt = DataLoader(train_ds_nt, batch_size=BATCH_SIZE, shuffle=True)
val_loader_nt = DataLoader(val_ds_nt, batch_size=BATCH_SIZE, shuffle=False)

model_no_timing = TimedGNN(NODE_FEAT_DIM, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT).to(device)
optimizer_nt = torch.optim.Adam(model_no_timing.parameters(), lr=LEARNING_RATE)

history_no_timing = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc_nt = -1.0

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc, _ = run_epoch(model_no_timing, train_loader_nt, optimizer_nt)
    val_loss, val_acc, _ = run_epoch(model_no_timing, val_loader_nt, optimizer=None)
    dt = time.time() - t0

    history_no_timing["train_loss"].append(train_loss)
    history_no_timing["val_loss"].append(val_loss)
    history_no_timing["train_acc"].append(train_acc)
    history_no_timing["val_acc"].append(val_acc)

    if val_acc > best_val_acc_nt:
        best_val_acc_nt = val_acc
        torch.save(model_no_timing.state_dict(), os.path.join(CKPT_DIR, "no_timing_gnn_best.pt"))

    print(f"[no-timing] epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} | {dt:.1f}s")

print(f"\nMiglior val accuracy (senza timing): {best_val_acc_nt:.3f} -> salvato in "
      f"{os.path.join(CKPT_DIR, 'no_timing_gnn_best.pt')}")


## 7. Curve di training (con vs senza timing)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_timed["val_loss"], label="con timing")
axes[0].plot(history_no_timing["val_loss"], label="senza timing")
axes[0].set_title("Validation loss")
axes[0].set_xlabel("epoca"); axes[0].set_ylabel("loss"); axes[0].legend()

axes[1].plot(history_timed["val_acc"], label="con timing")
axes[1].plot(history_no_timing["val_acc"], label="senza timing")
axes[1].set_title("Validation accuracy@1")
axes[1].set_xlabel("epoca"); axes[1].set_ylabel("accuracy"); axes[1].legend()

plt.tight_layout()
plt.show()


## 8. Valutazione sul test set interno + breakdown per `n`

Prima approssimazione della research question 1 ("per quali valori di n il modello timed va meglio").
Il confronto **vero** con l'LLM avverrà nella Fase 4 sul set held-out esterno; qui verifichiamo il
comportamento del modello GNN sul proprio test set interno.


In [ ]:
test_ds = ChessPuzzleGraphDataset(maybe_subset(df_test, SUBSET_SIZE // 5 if SUBSET_SIZE else None), include_timing=True)
test_ds_nt = ChessPuzzleGraphDataset(maybe_subset(df_test, SUBSET_SIZE // 5 if SUBSET_SIZE else None), include_timing=False)

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader_nt = DataLoader(test_ds_nt, batch_size=BATCH_SIZE, shuffle=False)

model_timed.load_state_dict(torch.load(os.path.join(CKPT_DIR, "timed_gnn_best.pt")))
model_no_timing.load_state_dict(torch.load(os.path.join(CKPT_DIR, "no_timing_gnn_best.pt")))

test_loss_t, test_acc_t, per_n_t = run_epoch(model_timed, test_loader, optimizer=None)
test_loss_nt, test_acc_nt, per_n_nt = run_epoch(model_no_timing, test_loader_nt, optimizer=None)

print(f"Test accuracy (con timing):    {test_acc_t:.3f}")
print(f"Test accuracy (senza timing):  {test_acc_nt:.3f}")
print(f"Delta (obiettivo 5, ablation): {test_acc_t - test_acc_nt:+.3f}")


In [ ]:
ns = sorted(set(per_n_t) | set(per_n_nt))
width = 0.35
x = np.arange(len(ns))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width/2, [per_n_t.get(n, 0) for n in ns], width, label="con timing")
ax.bar(x + width/2, [per_n_nt.get(n, 0) for n in ns], width, label="senza timing")
ax.set_xticks(x); ax.set_xticklabels([f"mate in {n}" for n in ns])
ax.set_ylabel("accuracy@1"); ax.set_title("Accuracy GNN per valore di n (test set interno)")
ax.legend()
plt.tight_layout()
plt.show()

print("\nTabella accuracy per n:")
for n in ns:
    print(f"  n={n}: con timing={per_n_t.get(n, float('nan')):.3f}  senza timing={per_n_nt.get(n, float('nan')):.3f}")


## Riepilogo Fase 3 e prossimi passi

**Cosa abbiamo costruito:**
- `TimedGNN`: encoder message-passing (`GINEConv`, 4 layer, residual + layer norm) + policy head su archi
  "mossa legale", con softmax raggruppata per grafo (gestisce correttamente il numero variabile di mosse
  candidate per posizione)
- Training loop completo con loss NLL, accuracy@1, e **accuracy per valore di `n`**
- Due modelli addestrati e salvati in `/content/checkpoints/`: `timed_gnn_best.pt` e
  `no_timing_gnn_best.pt`, pronti per l'ablation study (obiettivo 5)
- Prima analisi per `n` sul test set interno

**Cose da tarare prima del training "vero" (su tutto il dataset, 50-100 epoche):**
- `SUBSET_SIZE = None` per usare tutti i puzzle
- Eventualmente aumentare `hidden_dim` a 256 e i layer a 5, come indicato nella metodologia
- Su Colab gratuito, valutare checkpoint intermedi e riprese di training (i runtime si disconnettono)

**Prossimi passi:**
1. **Baseline LLM**: prompting su un modello (open-source o via API) sugli stessi puzzle, con lo stesso
   schema FEN + "trova il matto in n mosse"
2. **Set di validazione held-out** (esterno a Lichess, Fase 1 punto 8): applicare sia GNN che LLM e
   confrontare con McNemar's test / accuracy per n
3. **Report finale**: raccogliere risultati, grafici, insight (Objectives 4-6 del progetto)

Dimmi quando vuoi procedere con la baseline LLM (punto 1) e continuo da lì.
